# Part 2 Summary

Part Two of *Practical Deep Learning for Coders* spans nineteen lessons (numbered 9 through 25, plus two bonus sessions) and inverts Part One's direction entirely: where Part One started with a working model and peeled back the layers, Part Two starts from nothing — pure Python and the standard library — and builds upward until a complete Stable Diffusion pipeline exists from scratch. Jeremy half-jokingly calls it "Impractical Deep Learning for Coders," but the purpose is deeply practical: understanding the foundations well enough to do original research, customize production systems, and navigate a field that reinvents itself every few months.

Lesson 9 opens with a hands-on tour of Stable Diffusion through Hugging Face's Diffusers library. Guidance scale, negative prompts, image-to-image pipelines, textual inversion, and Dreambooth are demonstrated as practical tools before the underlying theory is touched. Then comes a conceptual explanation of diffusion that Jeremy presents as simpler than the standard derivation: imagine a function that returns the probability that an image is a handwritten digit — its gradient tells you which pixels to nudge, and repeatedly nudging transforms noise into a recognizable digit. The neural network doesn't need to learn the probability itself, only the noise that was added, and the training data is trivially generated by adding random noise to real images. The architecture for this noise prediction is a U-Net; the computational cost is managed by an autoencoder that compresses images into compact latents; and text conditioning arrives via CLIP, which trains paired image and text encoders so that semantically similar descriptions map to nearby embedding vectors. In Lesson 9A, Jonathan Whitaker disassembles the Stable Diffusion pipeline component by component, demonstrating token-level embedding manipulation and loss-based guidance. In Lesson 9B, Waseem and Tanishq walk through the original probability theory from the 2015 Sohl-Dickstein paper through the 2020 DDPM simplification, showing that the probabilistic and score-based perspectives converge on the same thing: predicting noise is learning the score function.

With diffusion conceptually grounded, Lesson 10 rewinds all the way to pure Python. Tensors are introduced as the natural descendant of APL arrays, built up from a custom `Matrix` class with `__getitem__`. Pseudo-random number generation is implemented from scratch (the Wichmann-Hill algorithm), and a critical bug is demonstrated live: `os.fork()` copies random state identically into child processes, producing correlated "random" numbers in parallel data loaders — a real bug that once affected fastai itself. Lesson 11 builds matrix multiplication four ways — naive Python loops, Numba-compiled inner loops, broadcasting, and Einstein summation — progressing from 450 milliseconds for 5 images to 656 milliseconds for 50,000, and finally 458 microseconds on a GPU: a five-million-fold cumulative speedup. Broadcasting, the mechanism that lets a vector operate across all rows of a matrix without explicit loops, is emphasized as the single most important foundational operation in deep learning code. Paper-reading skills are developed alongside: Jeremy demonstrates saving papers to Zotero, using LaTeX OCR to decode unfamiliar notation, downloading arXiv source to find symbol definitions, and reading the DiffEdit paper's mask-generation algorithm step by step. The Greek alphabet is treated as essential vocabulary.

Lesson 12 expands the mathematical toolkit through mean shift clustering, built from scratch as an exercise in broadcasting, Gaussian kernels, norms, and weighted averages. The algorithm is batched for GPU execution and collapses from 600 milliseconds to 1 millisecond. Calculus is introduced through the infinitesimal perspective — dy/dx as a ratio of very small real numbers — and the chain rule is animated with a GeoGebra demonstration of connected wheels.

Lesson 13 builds a forward and backward pass through a multi-layer perceptron from scratch, with gradient functions for MSE, linear layers, and ReLU derived by hand and verified against PyTorch's autograd. The Python debugger is introduced as one of the most powerful tools a programmer can learn. In Lesson 14, the code is refactored through progressively cleaner abstractions — standalone functions become callable classes, then classes inheriting from a custom `Module` base — until it mirrors `nn.Module` closely enough that the library version can be adopted with full understanding. Cross-entropy loss is derived from first principles: softmax converts raw outputs to probabilities, log softmax uses the log-sum-exp trick for numerical stability, and negative log likelihood reduces to a single array index into the correct class position. A training loop achieves 97% accuracy on MNIST in three epochs. The refactoring continues through datasets, data loaders, samplers, collation functions, and optimizers — each built from scratch, then replaced by PyTorch's version. Hugging Face Datasets are bridged to PyTorch's tuple-based expectations via a custom `collate_dict`. Python foundations — `__setattr__`, `__getattr__`, `yield from`, `*args`/`**kwargs`, decorators, context managers, f-string format specifiers, and eleven dunder methods — are taught not as isolated topics but as tools needed to understand the framework code being written.

Lesson 15 introduces convolutions as encoding spatial priors that nearby pixels are related and that the same pattern should be recognized regardless of location. The im2col trick converts convolutions into matrix multiplications, and stride-2 convolutions replace pooling for spatial reduction. A CNN achieves comparable accuracy to the MLP with one-eighth the parameters. Receptive fields are explored in Excel using Trace Precedents. A convolutional autoencoder on Fashion MNIST produces disappointing results, and the difficulty of training it properly motivates the construction of a flexible training framework.

The Learner class, callbacks, and metrics system that emerges in Lesson 16 becomes the scaffolding for everything that follows. Callbacks are introduced progressively — from plain functions through lambdas and `functools.partial` to callable objects with named lifecycle methods. Custom exceptions (`CancelFitException`, `CancelEpochException`, `CancelBatchException`) enable clean control flow. A `@contextmanager` decorator compresses the verbose try/except boilerplate into one-liners. The most striking design: the Learner's `predict`, `get_loss`, `backward`, `step`, and `zero_grad` methods don't exist on the Learner itself — `__getattr__` intercepts them and routes to callbacks, making the training logic entirely pluggable. PyTorch hooks provide the same callback concept for inspecting activations inside a model, and the "colorful dimension" histogram plot — a single image showing the full distribution of activations across all batches — becomes the primary diagnostic tool. If activations spike and crash, stop training: the model will probably never recover.

Initialization and normalization occupy Lesson 17. A 50-layer toy experiment demonstrates that naive random initialization produces either NaN or zero after repeated matrix multiplies. Glorot initialization scales by 1/√n to preserve variance, but breaks with ReLU; Kaiming initialization adds a √2 factor to compensate. General ReLU — leaky ReLU with a downward shift — improves statistics further. LSUV provides a completely general alternative: pass one batch through the model and iteratively adjust each layer's bias and weight scaling until mean and standard deviation converge to 0 and 1. Batch normalization, layer normalization, instance normalization, and group normalization are all implemented and compared. The recurring theme is restated: nearly every improvement in deep learning comes down to getting mean-zero, unit-variance inputs and activations.

Lesson 18 builds accelerated SGD in a spreadsheet before coding it. Momentum maintains an exponentially weighted moving average of gradients; RMSProp divides by a moving average of squared gradients to normalize step sizes; Adam combines both. A novel experiment right in the spreadsheet explores automatic learning rate annealing based on gradient statistics. PyTorch's cosine annealing and Leslie Smith's 1-Cycle policy are demonstrated, and a ResNet is built from scratch with skip connections, batch-norm-zero initialization, and a model summary showing parameter counts and megaflops per layer. Data augmentation — random crop, random horizontal flip, random erasing, and test time augmentation — pushes Fashion MNIST to 94.6% accuracy, competitive with the best published results.

Lesson 19 circles back to diffusion. Dropout is implemented from scratch (including the 1/(1-p) scaling for surviving activations and Dropout2d for channel-wise masking), and test-time dropout is demonstrated as a confidence measure. DDPM is built as a single miniai callback: the forward process noises clean images, the U-Net predicts noise, MSE provides the loss, and the reverse process iteratively denoises. After five minutes of training on Fashion MNIST, generated images show recognizable shirts, shoes, and pants — far surpassing what GANs achieved after hours. In Lesson 20, mixed precision training arrives as another callback, and HuggingFace's Accelerate library provides a higher-level alternative. Style transfer demonstrates perceptual loss and Gram matrices, and Neural Cellular Automata show that tiny networks with only 168 parameters can grow and self-repair complex textures.

Lesson 21 scales up to CIFAR-10 and introduces Weights & Biases for experiment tracking. FID and KID are implemented from scratch to measure generation quality, including a Newton-Schulz matrix square root. A debugging saga — switching from [0,1] to [-1,1] pixel range made results worse — leads to the discovery that [-0.5, 0.5] works dramatically better, and that Robin Rombach's mysterious 0.18 scaling factor in Stable Diffusion serves the same purpose. DDIM sampling enables deterministic generation with far fewer steps. In Lesson 22, the discrete step count T is eliminated in favor of a continuous noise schedule. The Karras et al. paper's unified framework simplifies everything further: preconditioning constants (c_skip, c_in, c_out) ensure unit-variance inputs and outputs regardless of noise level, and the Euler, Ancestral Euler, Heun, and LMS samplers each reduce to a handful of lines. Heun's method at 50 steps achieves FID 0.97 — nearly indistinguishable from real data.

Lesson 23 opens with a double bug fix in the previous notebook's FID measurement, then moves to Tiny ImageNet (64×64, 200 classes). A super-resolution UNet with perceptual loss, transfer learning, and cross connections produces visible improvements. Lesson 24 constructs a UNet from scratch with pre-activation convolutions, SaveModule mixins for automatic skip connections, sinusoidal time embeddings, and self-attention flattened from 2D feature maps to 1D sequences. Multi-head attention is implemented with einops `rearrange`, and the resulting Transformer block is verified against PyTorch's `nn.TransformerEncoder`. Class conditioning adds a single `nn.Embedding` to the time embedding pathway.

Lesson 25, the final lesson, covers three major topics. Audio generation treats bird calls as Mel spectrograms and feeds them through the same diffusion pipeline. The VAE is built on Fashion MNIST — KL divergence loss pushes the latent space toward a smooth, navigable standard normal — and then the pre-trained Stable Diffusion VAE encodes LSUN Bedrooms into memory-mapped latents for efficient training. A latent-space ImageNet classifier trained on VAE-encoded images reaches 66% accuracy — surpassing AlexNet — opening avenues for latent-CLIP, latent-perceptual-loss, and latent-FID that could make the entire diffusion pipeline cheaper. The course concludes with the promise of Part 3 covering NLP, GPT-style models, and the final piece: CLIP text conditioning to complete Stable Diffusion entirely from scratch.

---

**Lesson Challenges**

- Experiment with Stable Diffusion features: guidance scale, negative prompts, image-to-image, textual inversion (Lesson 9)
- Implement negative prompts, image-to-image, and callbacks in the from-scratch pipeline (Lesson 10)
- Implement Step 1 of DiffEdit: generate a mask by contrasting noise predictions from two text prompts (Lesson 11)
- GPU-accelerate DBSCAN, k-means, spectral clustering, or LSH; invent a new mean shift variant (Lesson 12)
- Recreate log_softmax, nll_loss, cross_entropy, and the full forward/backward pass from scratch (Lesson 13)
- Rebuild `nn.Module`, SGD optimizer, Dataset, DataLoader, Sampler, and collate function from scratch (Lesson 14)
- Implement im2col from scratch; build a convolutional autoencoder (Lesson 15)
- Build custom callbacks, the learning rate finder, and activation histogram diagnostics (Lesson 16)
- Get Fashion MNIST to ≥90% without ResNets; turn LSUV into a callback (Lessons 17)
- Beat 93.8% (20 epochs) or 94.6% (50 epochs) on Fashion MNIST (Lesson 18)
- Implement Dropout2d from scratch; try alternative noise schedules for DDPM (Lesson 19)
- Implement style transfer, Neural Cellular Automata, and mixed precision as callbacks (Lesson 20)
- Implement FID/KID from scratch; re-implement DDIM sampling (Lesson 21)
- Predict noise level without *t*; implement Karras preconditioning and samplers (Lesson 22)
- Build UNet super-resolution with perceptual loss; try segmentation, colorization, in-painting, watermark removal (Lesson 23)
- Build a diffusion UNet from scratch with attention and class conditioning (Lesson 24)
- Build audio diffusion, VAE, and latent diffusion from scratch (Lesson 25)

**Potential Research Directions**

- Treating diffusion inference as an optimization problem rather than ODE solving — applying optimizer concepts (momentum, Adam) to sampling (Lesson 9)
- Removing time step *t* as a UNet input — early experiments by Jeremy and Robert Rombach show promising results (Lesson 22)
- Using perceptual loss instead of MSE for training noise prediction (Lesson 9)
- Optimal input scaling for diffusion models — why [-0.5, 0.5] outperforms [-1, 1] (Lesson 21)
- Automatic learning rate annealing based on gradient statistics (Lesson 18)
- Proper initialization removing the need for batch norm and warmup (Fixup, T-Fixup) (Lesson 18)
- Pre-trained latent-space backbones, latent-CLIP, and latent-perceptual-loss (Lesson 25)
- Test-time dropout as a calibrated confidence/uncertainty measure (Lesson 19)
- Non-uniform timestep sampling during training (Lesson 20)
- Reducing or eliminating normalization layers while maintaining training stability (Lesson 17)
- Neural Cellular Automata for denoising, stylizing, and self-repair (Lesson 20)
- 2D-native attention vs. flattened 1D sequences in diffusion UNets (Lesson 24)
- Connections between the probabilistic (DDPM/ELBO) and score-based (score matching) perspectives (Lesson 9B)
- Random copying as an alternative to random erasing and its relationship to CutMix (Lesson 18)
- Better phase reconstruction (deep learning replacements for Griffin-Lim) for audio generation (Lesson 25)

**Homework**

- Play extensively with Stable Diffusion; run the `stable_diffusion.ipynb` notebook; explore community tools (Lesson 9)
- Read the PyTorch tensor documentation end-to-end; read the Python standard library docs for every method you use (Lesson 10)
- Practice broadcasting and einsum until second nature; learn the Greek alphabet (Lesson 11)
- Watch 3Blue1Brown's "Essence of Calculus" series (Lesson 12)
- Go through notebook 3 with cleared outputs, predicting shapes before running cells (Lesson 13)
- Study Python's Data Model documentation for dunder methods; use `pdb.set_trace()` liberally (Lesson 14)
- Practice Python foundations: try/except, decorators, getattr, properties, subclassing (Lesson 15)
- Create dummy versions of every unfamiliar Python feature; review standard deviations (Lesson 16)
- Experiment with General ReLU leak/subtraction values and different Adam β values (Lesson 17)
- Create custom schedulers; try to beat Jeremy's Fashion MNIST results using miniai (Lesson 18)
- Read the DDPM and Improved DDPM papers (Lesson 19)
- Run style transfer experiments with different layer combinations, loss ratios, and starting images (Lesson 20)
- Train DDPM_v3 with corrected input range; implement DDIM with varying η values (Lesson 21)
- Study Karras Appendix B6 derivations; compare samplers at different step counts (Lesson 22)
- Implement super-resolution, segmentation, colorization, in-painting, and watermark removal (Lesson 23)
- Complete the unconditional and conditional diffusion UNet notebooks (Lesson 24)
- Rebuild the audio diffusion and latent diffusion pipelines from scratch (Lesson 25)

**Things Jeremy Says You Should Do**

- Do Part 1 before Part 2 unless you're comfortable writing a basic SGD loop from scratch
- Spend ~10 hours per lesson; watch videos multiple times, pausing to look things up
- Read the Python documentation for every method you use and look at every option
- Read the PyTorch tensor documentation — scroll through the whole thing
- Use Shift+Tab, `?`, and `??` in Jupyter for signatures, docs, and source
- Learn to use the Python debugger (`pdb`) — one of the most powerful things you can do
- Learn the Greek alphabet so you can read equations aloud
- Use Zotero for paper management; read abstracts and results first, skip unhelpful sections
- Always test analytic gradients against finite differencing as a sanity check
- Always look at activation stats and colorful dimension plots — even for models you think are training well
- If you see the rising-crash pattern early in training, stop and restart
- Normalize your inputs — trivially easy and critically important
- Care about initialization — most people and frameworks don't
- Always draw pictures and visualize intermediate results — you get things wrong the first several attempts
- Build code step by step, checking shapes and values, then merge into functions
- Try to have all important code fit on one screen; use single-letter variables matching paper notation
- Build the absolute simplest version first and fully verify it before adding complexity
- Never use a mutable list as a default parameter — use a tuple
- Always question "standard" choices that lack theoretical justification
- Be very directed in experimentation — hypotheses over hyperparameter sweeps
- Use the smallest dataset possible for R&D until it's no longer challenging
- Call `clean_mem` for CUDA OOM errors instead of restarting the notebook
- Start every tabular/structured project with the simplest possible baseline
- Don't start by writing functions — write single lines you can run and check independently
- Consider learning APL — many say it taught them more about programming than anything else
- Join the fast.ai forums and Discord; share your work; ask for help

**Resources**

- [Course homepage](https://course.fast.ai) · [Part 2 overview](https://course.fast.ai/Lessons/part2.html)
- [fast.ai Forums](https://forums.fast.ai) · [fast.ai Discord](https://discord.gg/fastai)
- [course22p2 repo (notebooks and miniai)](https://github.com/fastai/course22p2)
- [diffusion-nbs repo](https://github.com/fastai/diffusion-nbs) — `stable_diffusion.ipynb` and Johno's Deep Dive notebook
- [fastbook (free book as notebooks)](https://github.com/fastai/fastbook) · [Amazon edition](https://www.amazon.com/Deep-Learning-Coders-fastai-PyTorch-ebook-dp-B08C2KM7NR/dp/B08C2KM7NR)
- **Lesson Videos:** [L9](https://youtu.be/_7rMfsA24Ls) · [L9A (Johno)](https://youtu.be/0_BBRNYInx8) · [L9B (Waseem & Tanishq)](https://youtu.be/mYpjmM7O-30) · [L10](https://course.fast.ai/Lessons/lesson10.html) · [L11](https://course.fast.ai/Lessons/lesson11.html) · [L12](https://course.fast.ai/Lessons/lesson12.html) · [L13](https://course.fast.ai/Lessons/lesson13.html) · [L14](https://course.fast.ai/Lessons/lesson14.html) · [L15](https://course.fast.ai/Lessons/lesson15.html) · [L16](https://course.fast.ai/Lessons/lesson16.html) · [L17](https://course.fast.ai/Lessons/lesson17.html) · [L18](https://course.fast.ai/Lessons/lesson18.html) · [L19](https://course.fast.ai/Lessons/lesson19.html) · [L20](https://course.fast.ai/Lessons/lesson20.html) · [L21](https://course.fast.ai/Lessons/lesson21.html) · [L22](https://course.fast.ai/Lessons/lesson22.html) · [L23](https://course.fast.ai/Lessons/lesson23.html) · [L24](https://course.fast.ai/Lessons/lesson24.html) · [L25](https://course.fast.ai/Lessons/lesson25.html)
- **Key Papers:** [DDPM (Ho et al., 2020)](https://arxiv.org/abs/2006.11239) · [DDIM (Song et al., 2020)](https://arxiv.org/abs/2010.02502) · [Improved DDPM (Nichol & Dhariwal, 2021)](https://arxiv.org/abs/2102.09672) · [Karras et al., "Elucidating the Design Space" (2022)](https://arxiv.org/abs/2206.00364) · [Ting Chen, "On the Importance of Noise Scheduling" (2023)](https://arxiv.org/abs/2301.10972) · [Sohl-Dickstein et al. (2015)](https://arxiv.org/abs/1503.03585) · [DiffEdit (2022)](https://arxiv.org/abs/2210.11427) · [Imagic (2022)](https://arxiv.org/abs/2210.09276) · [Progressive Distillation (Salimans & Ho)](https://arxiv.org/abs/2202.00512) · [Distillation of Guided Diffusion (Meng et al.)](https://arxiv.org/abs/2210.09671) · [Gatys et al., "A Neural Algorithm of Artistic Style"](https://arxiv.org/abs/1508.06576) · [Growing Neural Cellular Automata (Mordvintsev et al.)](https://distill.pub/2020/growing-ca/) · [U-Net (Ronneberger et al., 2015)](https://arxiv.org/abs/1505.04597) · [Deep Residual Learning (He et al., 2015)](https://arxiv.org/abs/1512.03385) · [Identity Mappings in ResNets (He et al., 2016)](https://arxiv.org/abs/1603.05027) · [Glorot & Bengio (2010)](http://proceedings.mlr.press/v9/glorot10a) · [Kaiming He et al., "Delving Deep into Rectifiers"](https://arxiv.org/abs/1502.01852) · [LSUV (Mishkin & Matas)](https://arxiv.org/abs/1511.06422) · [Batch Normalization (Ioffe & Szegedy)](https://arxiv.org/abs/1502.03167) · [Layer Normalization (Ba et al.)](https://arxiv.org/abs/1607.06450) · [Group Normalization (Wu & He)](https://arxiv.org/abs/1803.08494) · [Cyclical Learning Rates (Leslie Smith)](https://arxiv.org/abs/1506.01186) · [Fixup Initialization](https://arxiv.org/abs/1901.09321) · [TrivialAugment](https://arxiv.org/abs/2103.10158) · [ViT — "An Image is Worth 16x16 Words"](https://arxiv.org/abs/2010.11929) · [The Matrix Calculus You Need For Deep Learning (Parr & Howard)](https://explained.ai/matrix-calculus/)
- **Libraries & Tools:** [Hugging Face Diffusers](https://huggingface.co/docs/diffusers) · [Hugging Face Accelerate](https://huggingface.co/docs/accelerate) · [Hugging Face Datasets](https://huggingface.co/docs/datasets/) · [Weights & Biases](https://wandb.ai/) · [einops](https://github.com/arogozhnikov/einops) · [Numba](https://numba.pydata.org/) · [timm](https://github.com/huggingface/pytorch-image-models) · [Kat Crowson's k-diffusion](https://github.com/crowsonkb/k-diffusion) · [nbdev](https://nbdev.fast.ai/) · [fastcore](https://github.com/fastai/fastcore) · [fastprogress](https://github.com/fastai/fastprogress) · [torcheval](https://pytorch.org/torcheval/) · [Zotero](https://www.zotero.org/) · [pix2tex / LaTeX-OCR](https://github.com/lukas-blecher/LaTeX-OCR) · [Detexify](https://detexify.kirelabs.org/classify.html) · [TryAPL](https://tryapl.org/)
- **Spreadsheets:** [Accelerated SGD (graddesc.xlsm)](https://github.com/fastai/course22p2/blob/master/xl/graddesc.xlsm) · [Convolutions (conv-example.xlsx)](https://github.com/fastai/course22/tree/master/xl)
- **Other:** [Lexica](https://lexica.art/) · [PromptHero](https://prompthero.com/) · [Riffusion](https://www.riffusion.com/) · [Distill.pub Feature Visualization](https://distill.pub/2017/feature-visualization/) · [3Blue1Brown — Essence of Calculus](https://www.youtube.com/watch?v=WUvTyaaNkzM) · [Khan Academy — Calculus](https://www.khanacademy.org/math/calculus-1) · [Greek alphabet](https://en.wikipedia.org/wiki/Greek_alphabet) · [Notation as a Tool of Thought (Ken Iverson)](https://www.jsoftware.com/papers/tot.htm) · [Papers with Code — Fashion MNIST](https://paperswithcode.com/sota/image-classification-on-fashion-mnist)
